In [1]:
import numpy as np
import pandas as pd
import gmsh
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx.fem.petsc import NonlinearProblem, LinearProblem
import ufl
from dolfinx import fem, log, mesh, default_scalar_type
from dolfinx.fem.petsc import NonlinearProblem
from dolfinx.nls.petsc import NewtonSolver
from dolfinx.io import gmshio

In [2]:
# Material Parameters 
MU = 1.0
KAPPA = 10.0
PULL_STEPS = [0.001, 0.002, 0.005, 0.01, 0.05, 0.10]
MESH_SIZE = 0.02  # Matches the spacing in the DEM script
OUTPUT_FILE = '/Users/murat/Desktop/DEM-FEA-complex-geometry-comparison/fea_fenicsx_results.csv'

In [3]:
def create_gmsh_geometry(shape_name):
    """Programmatically generates the mesh using Gmsh OpenCASCADE."""
    gmsh.initialize()
    gmsh.clear()
    gmsh.model.add(shape_name)
    
    L, H = 1.0, 0.5
    
    # 1. Build geometries using robust OCC boolean operations
    if shape_name == 'rectangle':
        gmsh.model.occ.addRectangle(0, 0, 0, L, H, tag=1)
        
    elif shape_name == 'rectangle_hole':
        gmsh.model.occ.addRectangle(0, 0, 0, L, H, tag=1)
        gmsh.model.occ.addDisk(0.5, 0.25, 0, 0.1, 0.1, tag=2)
        gmsh.model.occ.cut([(2, 1)], [(2, 2)]) # Cut Object 2 from Object 1
        
    elif shape_name in ['l_shape', 'l_shape_hole']:
        # Base L-shape: 1x1 square minus 0.5x0.5 square at top right
        gmsh.model.occ.addRectangle(0, 0, 0, 1.0, 1.0, tag=1)
        gmsh.model.occ.addRectangle(0.5, 0.5, 0, 0.5, 0.5, tag=2)
        gmsh.model.occ.cut([(2, 1)], [(2, 2)], tag=3)
        domain_tag = 3
        
        # Add hole if requested
        if shape_name == 'l_shape_hole':
            gmsh.model.occ.addDisk(0.25, 0.25, 0, 0.1, 0.1, tag=4)
            gmsh.model.occ.cut([(2, domain_tag)], [(2, 4)])

    gmsh.model.occ.synchronize()
    
    # 2. Define the Physical Group (Required for FEniCSx import)
    surfaces = gmsh.model.getEntities(dim=2)
    surface_tags = [s[1] for s in surfaces]
    gmsh.model.addPhysicalGroup(2, surface_tags, tag=1)
    gmsh.model.setPhysicalName(2, 1, "Domain")

    # 3. Meshing
    gmsh.option.setNumber("Mesh.CharacteristicLengthMin", MESH_SIZE)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMax", MESH_SIZE)
    gmsh.model.mesh.generate(2)
    
    # Import Gmsh model directly into FEniCSx
    domain, _, _ = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, 0, gdim=2)
    gmsh.finalize()
    
    return domain

In [4]:
def solve_fea_shape(shape_name):
    print(f"\n--- Running FEA for {shape_name} ---")
    domain = create_gmsh_geometry(shape_name)
    
    # Define Vector Function Space for Displacement
    V = fem.functionspace(domain, ("Lagrange", 1, (domain.geometry.dim,)))
    # Define Scalar Function Space for Stress Projection
    V_scalar = fem.functionspace(domain, ("Lagrange", 1))
    
    u = fem.Function(V)  # Displacement field
    v = ufl.TestFunction(V)
    
    # --- Kinematics ---
    d = len(u)
    I = ufl.variable(ufl.Identity(d))
    F = I + ufl.grad(u)
    C = F.T * F
    B = F * F.T  # Left Cauchy-Green
    
    # Invariants
    I1 = ufl.tr(C)
    J = ufl.det(F)
    
    # --- Strain Energy Density (Neo-Hookean) ---
    psi = (MU / 2.0) * (I1 - 2.0) - MU * ufl.ln(J) + (KAPPA / 2.0) * (ufl.ln(J))**2
    Pi = psi * ufl.dx
    
    # Derive Residual and Jacobian automatically via UFL
    F_form = ufl.derivative(Pi, u, v)
    J_form = ufl.derivative(F_form, u, ufl.TrialFunction(V))

    # --- Analytical Stress Definitions (UFL) ---
    sigma = (1.0 / J) * (MU * (B - I) + KAPPA * ufl.ln(J) * I)
    
    sig_xx = sigma[0, 0]
    sig_yy = sigma[1, 1]
    sig_xy = sigma[0, 1]
    von_mises = ufl.sqrt(sig_xx**2 - sig_xx*sig_yy + sig_yy**2 + 3.0*sig_xy**2)
    
    # --- NEW: Helper Function for L2 Projection ---
    # We use this instead of fem.Expression to robustly map stresses to nodes
    def project_to_nodes(expr):
        u_t = ufl.TrialFunction(V_scalar)
        v_t = ufl.TestFunction(V_scalar)
        a = u_t * v_t * ufl.dx
        L = expr * v_t * ufl.dx
        # LU factorization is extremely fast for this mass matrix
        problem = LinearProblem(a, L, bcs=[], petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
        return problem.solve()

    # --- Boundary Conditions ---
    coords = domain.geometry.x
    x_max = np.max(coords[:, 0])
    
    def left_boundary(x):
        return np.isclose(x[0], 0.0)
    
    def right_boundary(x):
        return np.isclose(x[0], x_max)

    fdim = domain.topology.dim - 1
    left_facets = mesh.locate_entities_boundary(domain, fdim, left_boundary)
    right_facets = mesh.locate_entities_boundary(domain, fdim, right_boundary)
    
    left_dofs = fem.locate_dofs_topological(V, fdim, left_facets)
    right_dofs = fem.locate_dofs_topological(V, fdim, right_facets)
    
    u_left = fem.Function(V)
    u_left.x.array[:] = 0.0  
    
    u_right = fem.Function(V) 
    
    bc_left = fem.dirichletbc(u_left, left_dofs)
    bc_right = fem.dirichletbc(u_right, right_dofs)
    bcs = [bc_left, bc_right]

    # --- Non-linear Solver Setup ---
    problem = NonlinearProblem(F_form, u, bcs, J_form)
    solver = NewtonSolver(domain.comm, problem)
    solver.atol = 1e-8
    solver.rtol = 1e-8
    solver.convergence_criterion = "incremental"

    dof_coords = V.tabulate_dof_coordinates()
    num_dofs = dof_coords.shape[0]
    shape_results = []

    # --- Load Stepping ---
    for pull in PULL_STEPS:
        print(f"Solving pull step: {pull}...")
        
        def right_displacement(x):
            values = np.zeros((2, x.shape[1]), dtype=default_scalar_type)
            values[0] = pull
            values[1] = 0.0
            return values
            
        u_right.interpolate(right_displacement)
        
        try:
            num_its, converged = solver.solve(u)
            print(f"  Converged in {num_its} iterations.")
            
            # --- NEW: Project stresses AFTER nonlinear solver converges ---
            func_sig_xx = project_to_nodes(sig_xx)
            func_sig_yy = project_to_nodes(sig_yy)
            func_sig_xy = project_to_nodes(sig_xy)
            func_von_mises = project_to_nodes(von_mises)
            
        except RuntimeError:
            print("  Newton solver failed to converge! Skipping remaining pulls.")
            break
            
        # --- Data Extraction ---
        displacements = u.x.array.reshape((num_dofs, 2))
        
        s_xx = func_sig_xx.x.array
        s_yy = func_sig_yy.x.array
        s_xy = func_sig_xy.x.array
        vm = func_von_mises.x.array
        
        for i in range(num_dofs):
            shape_results.append({
                'Shape': shape_name,
                'Pull': float(pull),
                'X': float(dof_coords[i, 0]),
                'Y': float(dof_coords[i, 1]),
                'u_x': float(displacements[i, 0]),
                'u_y': float(displacements[i, 1]),
                'sigma_xx': float(s_xx[i]),
                'sigma_yy': float(s_yy[i]),
                'sigma_xy': float(s_xy[i]),
                'von_mises': float(vm[i])
            })
            
    return shape_results

In [5]:
import os
import pandas as pd
from dolfinx import log

# Suppress excessive FEniCSx solver logs
log.set_log_level(log.LogLevel.WARNING)

shapes = ['rectangle', 'rectangle_hole', 'l_shape', 'l_shape_hole']
all_fea_results = []
    
for shape in shapes:
    results = solve_fea_shape(shape)  # Make sure this calls solve_fea_shape!
    all_fea_results.extend(results)
        
df_fea = pd.DataFrame(all_fea_results)

# --- NEW CODE: Force create the directory structure ---
output_dir = os.path.dirname(OUTPUT_FILE)
if output_dir:  
    os.makedirs(output_dir, exist_ok=True)
# ----------------------------------------------------

df_fea.to_csv(OUTPUT_FILE, index=False)
print(f"\nFEA Simulation complete! Extracted {len(df_fea)} nodal data points to '{OUTPUT_FILE}'.")


--- Running FEA for rectangle ---
Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.000159125s, CPU 0.000176s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0248606s, CPU 0.024217s)
Info    : 1539 nodes 3080 elements
Solving pull step: 0.001...
  Converged in 3 iterations.


ld: warning: duplicate -rpath '/opt/miniconda3/envs/fenicsx-env/lib' ignored
ld: warning: duplicate -rpath '/opt/miniconda3/envs/fenicsx-env/lib' ignored
ld: warning: duplicate -rpath '/opt/miniconda3/envs/fenicsx-env/lib' ignored
ld: warning: duplicate -rpath '/opt/miniconda3/envs/fenicsx-env/lib' ignored
ld: warning: duplicate -rpath '/opt/miniconda3/envs/fenicsx-env/lib' ignored


Solving pull step: 0.002...
  Converged in 3 iterations.
Solving pull step: 0.005...
  Converged in 3 iterations.
Solving pull step: 0.01...
  Converged in 3 iterations.
Solving pull step: 0.05...
  Converged in 4 iterations.
Solving pull step: 0.1...
  Converged in 4 iterations.

--- Running FEA for rectangle_hole ---
Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Meshing 1D...                                                                                                                        
Info    : [  0%] Meshing curve 5 (Ellipse)
Info    : [ 30%] Meshing curve 6 (Line)
Info    : [ 50%] Meshing curve 7 (Line)
Info    : [ 70%] Meshing curve 8 (Line)
Info    : [ 90%] Meshing curve 9 (Line)
Info    : Done meshing 1D (Wall 0.000161s, CPU 0.000179s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0163325s, CPU 0.016145s)
Info    : 1509 nodes 3023 elements
Solving pull s